# Week 2.3: Embeddings Deep Dive

## Key Insight
> **How computers turn words into numbers (Vectors) to find 'meaning'**

## Learning Objectives
By the end of this notebook, you will:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

## Exercises
This notebook contains 5 exercises ranging from easy to advanced.

## Digital Twin Integration
This notebook extends your digital twin with production-ready capabilities from Week 2.

In [1]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Tuple
import re
import json

print('✅ Week 2.3: Embeddings Deep Dive - Ready to start!')
print('\nTopics covered:')
print('  - Word2Vec and GloVe internals')
print('  - Building embeddings from scratch')
print('  - Semantic similarity and vector operations')
print('  - Dimensionality reduction and visualization')
print('  - Using pre-trained embeddings')


✅ Week 2.3: Embeddings Deep Dive - Ready to start!

Topics covered:
  - Word2Vec and GloVe internals
  - Building embeddings from scratch
  - Semantic similarity and vector operations
  - Dimensionality reduction and visualization
  - Using pre-trained embeddings


---
## Part 1: Introduction to Word2Vec and GloVe internals

Let's start by understanding the fundamentals and building our first implementation.

In [3]:
from typing import List
import re

class SimpleTokenizer:
    """
    A basic tokenizer and vocabulary builder for preparing text for Word2Vec/GloVe models.
    """
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0
        self.word_counts = {}

    def clean_text(self, text: str) -> list[str]:
        """Lowercase text and remove punctuation, returning a list of tokens."""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def fit(self, corpus: List[str]):
        """Build vocabulary from a list of sentences."""
        for sentence in corpus:
            tokens = self.clean_text(sentence)
            for token in tokens:
                self.word_counts[token] = self.word_counts.get(token, 0) + 1
                if token not in self.word2idx:
                    self.word2idx[token] = self.vocab_size
                    self.idx2word[self.vocab_size] = token
                    self.vocab_size += 1

    def encode(self, sentence: str) -> List[int]:
        """Convert text into a list of vocabulary indices."""
        tokens = self.clean_text(sentence)
        return [self.word2idx[token] for token in tokens if token in self.word2idx]

    def decode(self, indices: List[int]) -> str:
        """Convert list of indices back to a sentence."""
        return " ".join([self.idx2word[idx] for idx in indices if idx in self.idx2word])

# Example usage
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "We are diving deep into embeddings and vector operations!"
]

tokenizer = SimpleTokenizer()
tokenizer.fit(corpus)

print(f"Vocabulary Size: {tokenizer.vocab_size}")
print(f"Word to Index Mapping: {tokenizer.word2idx}")

test_sentence = "The lazy fox jumps!"
encoded = tokenizer.encode(test_sentence)
decoded = tokenizer.decode(encoded)
print(f"\nOriginal: {test_sentence}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")

Vocabulary Size: 17
Word to Index Mapping: {'the': 0, 'quick': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'over': 5, 'lazy': 6, 'dog': 7, 'we': 8, 'are': 9, 'diving': 10, 'deep': 11, 'into': 12, 'embeddings': 13, 'and': 14, 'vector': 15, 'operations': 16}

Original: The lazy fox jumps!
Encoded:  [0, 6, 3, 4]
Decoded:  the lazy fox jumps


### 🎯 Exercise 1 (Easy): Getting Started

**Task**: Implement a basic version of the concept introduced above.

**Requirements**:
1. Follow the pattern shown in the example
2. Test your implementation with sample data
3. Verify the output matches expectations
4. Add error handling for edge cases

In [4]:
class CustomTokenizer:
    """
    A customizable tokenizer that builds a vocabulary from text corpus
    with support for minimum word frequency filtering and basic error handling.
    """
    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.word2idx = {"<UNK>": 0}
        self.idx2word = {0: "<UNK>"}
        self.vocab_size = 1
        self.word_counts = {}

    def clean_text(self, text: str) -> List[str]:
        if not isinstance(text, str):
            raise ValueError("Input must be a string.")
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def fit(self, corpus: List[str]):
        if not isinstance(corpus, list):
            raise TypeError("Corpus must be a list of strings.")

        # First, count all word frequencies
        for sentence in corpus:
            if not isinstance(sentence, str):
                continue
            tokens = self.clean_text(sentence)
            for token in tokens:
                self.word_counts[token] = self.word_counts.get(token, 0) + 1

        # Build vocabulary using words that meet the min_freq threshold
        for token, count in self.word_counts.items():
            if count >= self.min_freq and token not in self.word2idx:
                self.word2idx[token] = self.vocab_size
                self.idx2word[self.vocab_size] = token
                self.vocab_size += 1

    def encode(self, sentence: str) -> List[int]:
        try:
            tokens = self.clean_text(sentence)
            # Use '<UNK>' index (0) if the token is not found in vocabulary
            return [self.word2idx.get(token, 0) for token in tokens]
        except ValueError as e:
            print(f"Encoding error: {e}")
            return []

    def decode(self, indices: List[int]) -> str:
        if not isinstance(indices, list):
            return ""
        return " ".join([self.idx2word.get(idx, "<UNK>") for idx in indices])

# Test our custom implementation with a min_freq threshold of 2
sample_corpus = [
    "deep learning and deep neural networks",
    "learning embeddings is deep fun"
]

custom_tokenizer = CustomTokenizer(min_freq=2)
custom_tokenizer.fit(sample_corpus)

print(f"Vocabulary with min_freq=2: {custom_tokenizer.word2idx}")
print(f"Word Counts: {custom_tokenizer.word_counts}")

test_str = "deep neural networks are fun"
encoded_seq = custom_tokenizer.encode(test_str)
print(f"\nEncoded sequence: {encoded_seq}")
print(f"Decoded sequence: {custom_tokenizer.decode(encoded_seq)}")

Vocabulary with min_freq=2: {'<UNK>': 0, 'deep': 1, 'learning': 2}
Word Counts: {'deep': 3, 'learning': 2, 'and': 1, 'neural': 1, 'networks': 1, 'embeddings': 1, 'is': 1, 'fun': 1}

Encoded sequence: [1, 0, 0, 0, 0]
Decoded sequence: deep <UNK> <UNK> <UNK> <UNK>


---
## Part 2: Building embeddings from scratch

Building on the fundamentals, let's explore more advanced topics and implementations.

In [13]:
from typing import List
import re
import json

# Advanced implementation example

def advanced_function(data: list, **kwargs) -> dict[str]:
    """
    Advanced function for Building embeddings from scratch.

    Args:
        data: Input data to process
        **kwargs: Additional parameters

    Returns:
        dict: Processed results
    """
    results = {
        'processed': len(data),
        'data': data
    }
    return results

# Example usage
sample_data = ['example1', 'example2', 'example3']
output = advanced_function(sample_data)
print(json.dumps(output, indent=2))

{
  "processed": 3,
  "data": [
    "example1",
    "example2",
    "example3"
  ]
}


### 🎯 Exercise 2 (Intermediate): Building on the Basics

**Task**: Create a more sophisticated implementation that combines multiple concepts.

**Requirements**:
1. Use the patterns from Part 1 and Part 2
2. Add parameter validation
3. Include helpful error messages
4. Test with various input types

In [4]:
from typing import List, Tuple
import re

class CustomTokenizer:
    """
    A customizable tokenizer that builds a vocabulary from text corpus
    with support for minimum word frequency filtering and basic error handling.
    """
    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.word2idx = {"<UNK>": 0}
        self.idx2word = {0: "<UNK>"}
        self.vocab_size = 1
        self.word_counts = {}

    def clean_text(self, text: str) -> List[str]:
        if not isinstance(text, str):
            raise ValueError("Input must be a string.")
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def fit(self, corpus: List[str]):
        if not isinstance(corpus, list):
            raise TypeError("Corpus must be a list of strings.")

        # First, count all word frequencies
        for sentence in corpus:
            if not isinstance(sentence, str):
                continue
            tokens = self.clean_text(sentence)
            for token in tokens:
                self.word_counts[token] = self.word_counts.get(token, 0) + 1

        # Build vocabulary using words that meet the min_freq threshold
        for token, count in self.word_counts.items():
            if count >= self.min_freq and token not in self.word2idx:
                self.word2idx[token] = self.vocab_size
                self.idx2word[self.vocab_size] = token
                self.vocab_size += 1

    def encode(self, sentence: str) -> List[int]:
        try:
            tokens = self.clean_text(sentence)
            return [self.word2idx.get(token, 0) for token in tokens]
        except ValueError as e:
            print(f"Encoding error: {e}")
            return []

    def decode(self, indices: List[int]) -> str:
        if not isinstance(indices, list):
            return ""
        return " ".join([self.idx2word.get(idx, "<UNK>") for idx in indices])


class SkipGramDatasetGenerator:
    """
    Generates (target, context) word index pairs for training a Skip-Gram model
    from raw token sequences using a sliding context window.
    """
    def __init__(self, window_size: int = 2):
        if not isinstance(window_size, int) or window_size < 1:
            raise ValueError("Window size must be a positive integer.")
        self.window_size = window_size

    def generate_pairs(self, token_ids: List[int]) -> List[Tuple[int, int]]:
        """
        Generate (target, context) training pairs from a list of token IDs.
        """
        if not isinstance(token_ids, list):
            raise TypeError("Input must be a list of token IDs.")

        pairs = []
        n_tokens = len(token_ids)

        for i, target_id in enumerate(token_ids):
            # Boundary conditions: compute start and end indices of the window
            start = max(0, i - self.window_size)
            end = min(n_tokens, i + self.window_size + 1)

            for j in range(start, end):
                if j != i:  # Avoid target pairing with itself
                    pairs.append((target_id, token_ids[j]))
        return pairs

# Test implementation with our custom tokenizer output
dataset_generator = SkipGramDatasetGenerator(window_size=2)

# Instantiate CustomTokenizer directly here to make the cell self-contained
custom_tokenizer = CustomTokenizer(min_freq=1)

# Prepare sample data
corpus_for_word2vec = ["deep learning embeddings represent semantic meaning"]
custom_tokenizer.fit(corpus_for_word2vec)
encoded_tokens = custom_tokenizer.encode(corpus_for_word2vec[0])

# Generate pairs
training_pairs = dataset_generator.generate_pairs(encoded_tokens)

print(f"Encoded Tokens: {encoded_tokens}")
print(f"Token sequence decoded: {[custom_tokenizer.idx2word[i] for i in encoded_tokens]}")
print(f"Generated Skip-Gram Pairs (Target Index, Context Index):")
for target, context in training_pairs[:8]:
    target_word = custom_tokenizer.idx2word[target]
    context_word = custom_tokenizer.idx2word[context]
    print(f"  {target} ({target_word}) -> {context} ({context_word})")

Encoded Tokens: [1, 2, 3, 4, 5, 6]
Token sequence decoded: ['deep', 'learning', 'embeddings', 'represent', 'semantic', 'meaning']
Generated Skip-Gram Pairs (Target Index, Context Index):
  1 (deep) -> 2 (learning)
  1 (deep) -> 3 (embeddings)
  2 (learning) -> 1 (deep)
  2 (learning) -> 3 (embeddings)
  2 (learning) -> 4 (represent)
  3 (embeddings) -> 1 (deep)
  3 (embeddings) -> 2 (learning)
  3 (embeddings) -> 4 (represent)


---
## Part 3: Semantic similarity and vector operations

Now let's apply these concepts to real-world scenarios in your digital twin project.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt

def visualize_results(data: List[Any]):
    """
    Visualize the processing results.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Add your visualization code here
    ax.set_title('Results Visualization')
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')

    plt.tight_layout()
    plt.show()

# Example: visualize_results(sample_data)

### 🎯 Exercise 3 (Advanced): Complete Implementation

**Task**: Build a production-ready version that integrates with your digital twin.

**Requirements**:
1. Combine all concepts from this notebook
2. Add comprehensive error handling
3. Include logging and debugging capabilities
4. Create visualization of results
5. Write documentation for your implementation

In [5]:
from typing import List, Dict, Any
import numpy as np
import re

class CustomTokenizer:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0

    def fit(self, corpus: List[str]):
        for sentence in corpus:
            tokens = sentence.lower().split()
            for token in tokens:
                token = re.sub(r'[^a-z]', '', token)
                if token and token not in self.word2idx:
                    self.word2idx[token] = self.vocab_size
                    self.idx2word[self.vocab_size] = token
                    self.vocab_size += 1

class ProductionSystem:
    """
    Production-ready semantic system that implements word embeddings initialization,
    cosine similarity calculations, and semantic vector operations.
    """
    def __init__(self, embedding_dim: int = 50):
        if not isinstance(embedding_dim, int) or embedding_dim < 2:
            raise ValueError("Embedding dimension must be an integer >= 2.")
        self.embedding_dim = embedding_dim
        self.tokenizer = CustomTokenizer()
        self.embeddings = None

    def fit_and_initialize(self, corpus: List[str]):
        """
        Tokenize text corpus and initialize random embedding vectors for the vocabulary.
        """
        self.tokenizer.fit(corpus)
        if self.tokenizer.vocab_size == 0:
            raise ValueError("Corpus must contain valid alphabetical words.")

        # Initialize embeddings with normalized values
        np.random.seed(42)
        raw_embeddings = np.random.uniform(-0.5, 0.5, (self.tokenizer.vocab_size, self.embedding_dim))
        # Normalize each vector to unit length for easy cosine similarity calculations
        norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
        self.embeddings = raw_embeddings / np.where(norms == 0, 1, norms)

    def get_vector(self, word: str) -> np.ndarray:
        """Retrieve the embedding vector for a given word."""
        word_clean = re.sub(r'[^a-z]', '', word.lower())
        if word_clean not in self.tokenizer.word2idx:
            raise KeyError(f"Word '{word}' not found in vocabulary.")
        idx = self.tokenizer.word2idx[word_clean]
        return self.embeddings[idx]

    def cosine_similarity(self, word_a: str, word_b: str) -> float:
        """Compute cosine similarity between two words."""
        try:
            vec_a = self.get_vector(word_a)
            vec_b = self.get_vector(word_b)
            return float(np.dot(vec_a, vec_b))
        except KeyError as e:
            print(f"Similarity calculation error: {e}")
            return 0.0

    def find_most_similar(self, vector: np.ndarray, top_k: int = 3) -> List[Tuple[str, float]]:
        """Find top_k words most similar to a given target vector."""
        if self.embeddings is None:
            return []

        # Normalize target vector
        norm = np.linalg.norm(vector)
        target_norm = vector / (norm if norm != 0 else 1)

        similarities = np.dot(self.embeddings, target_norm)
        sorted_indices = np.argsort(similarities)[::-1]

        results = []
        for idx in sorted_indices[:top_k]:
            results.append((self.tokenizer.idx2word[idx], float(similarities[idx])))
        return results

# Run simulation tests
corpus = [
    "king queen prince man woman boy girl",
    "embeddings computer technology neural networks artificial intelligence",
    "digital twin python semantic operations similarity vector optimization"
]

system = ProductionSystem(embedding_dim=8)
system.fit_and_initialize(corpus)

# Test vector retrieval and cosine similarity
sim_val = system.cosine_similarity("king", "queen")
print(f"Vocabulary size: {system.tokenizer.vocab_size}")
print(f"Cosine similarity between 'king' and 'queen': {sim_val:.4f}")

# Try vector analogy simulation: king - man + woman
try:
    vec_king = system.get_vector("king")
    vec_man = system.get_vector("man")
    vec_woman = system.get_vector("woman")

    analogy_vector = vec_king - vec_man + vec_woman
    similar_words = system.find_most_similar(analogy_vector, top_k=3)
    print(f"\nAnalogy 'king' - 'man' + 'woman' matches:")
    for word, score in similar_words:
        print(f"  - {word}: similarity score = {score:.4f}")
except Exception as e:
    print(f"Error performing vector operations: {e}")

Vocabulary size: 22
Cosine similarity between 'king' and 'queen': 0.0292

Analogy 'king' - 'man' + 'woman' matches:
  - king: similarity score = 0.8236
  - woman: similarity score = 0.6729
  - similarity: similarity score = 0.5186


---
## Summary & Key Takeaways

### What We Learned:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

### Why This Matters:
- Essential for building production-ready systems
- Enables your digital twin to handle real-world scenarios
- Foundation for advanced topics in upcoming notebooks

### Next Steps:
1. Complete all exercises in this notebook
2. Integrate concepts with your digital twin project
3. Test your implementation thoroughly
4. Move on to the next notebook in Week 2

## 🏆 Challenge Project

**Build a Complete Word2Vec and GloVe internals System for Your Digital Twin**

**Requirements**:
1. Apply all concepts from this notebook
2. Handle edge cases and errors gracefully
3. Include comprehensive testing
4. Create visualizations of your results
5. Document your implementation
6. Integrate with your existing digital twin code

**Bonus Challenges**:
- Optimize for performance
- Add advanced features beyond the basics
- Create a user-friendly interface
- Write unit tests for your code